# FreshCart Supermarket Sales Insight — Complete Project
## Rounds 1–4: Data Preparation, EDA, Feature Engineering, Modeling, Excel Analysis & Pitch

**Use case:** Use Case 12 — Supermarket Sales Insight (FreshCart Superstores)  
**Purpose:** Answer the Round 2 requirements with a reproducible Google Colab notebook.

### What this notebook does
1. Creates a reproducible **synthetic practice dataset** if you don't upload the ZIP.
2. Loads and inspects the five CSV files.
3. Checks missing values, duplicates, invalid values, and relationships.
4. Cleans and combines sales, products, stores, promotions, and calendar data.
5. Performs EDA for category revenue, store trends, promotion impact, festival demand, seasonality, and peak periods.
6. Engineers features, encodes selected categorical variables, and performs preliminary feature selection.
7. Saves the cleaned/master dataset, engineered dataset, encoded dataset, charts, and a summary report.

**Important:** All records generated here are synthetic and are not real FreshCart business data. The promotion comparison is descriptive, not proof of causation.

### How to use
Run the notebook from top to bottom in Google Colab. No manual editing should be required. You may upload your own compatible `FreshCart_Synthetic_Dataset.zip` when prompted; choose **Cancel/skip** if you want the notebook to generate a fresh synthetic dataset automatically.

In [ ]:
# 0. Imports and configuration
import os
import io
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
SEED = 42
rng = np.random.default_rng(SEED)

BASE_DIR = Path("/content/freshcart_project") if Path("/content").exists() else (Path.cwd() / "freshcart_project")
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"
CHART_DIR = OUTPUT_DIR / "charts"
for folder in [DATA_DIR, OUTPUT_DIR, CHART_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Environment ready.")
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)

## 1. Get the datasets
If you have the ZIP file from the previous steps, upload it when prompted. If you skip the upload or close the picker, this notebook creates a synthetic dataset with the same five-file structure described in the use case.

In [ ]:
# 1A. Optional ZIP upload
# If you upload a ZIP, it should contain sales.csv, products.csv, stores.csv,
# promotions.csv, and calendar.csv. Otherwise, synthetic files are generated.

uploaded_zip = None
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        zip_candidates = [name for name in uploaded if name.lower().endswith(".zip")]
        if zip_candidates:
            uploaded_zip = zip_candidates[0]
            with zipfile.ZipFile(io.BytesIO(uploaded[uploaded_zip]), "r") as zf:
                zf.extractall(DATA_DIR)
            print(f"Extracted {uploaded_zip} into {DATA_DIR}")
        else:
            print("No ZIP selected. A synthetic dataset will be generated.")
    else:
        print("Upload skipped. A synthetic dataset will be generated.")
except Exception as exc:
    print("Upload is unavailable in this environment; generating synthetic data.", exc)

# Locate CSVs even if ZIP extraction created a nested folder
def locate_csv(filename, root=DATA_DIR):
    matches = list(Path(root).rglob(filename))
    return matches[0] if matches else None

required_files = ["sales.csv", "products.csv", "stores.csv", "promotions.csv", "calendar.csv"]
missing = [f for f in required_files if locate_csv(f) is None]

if missing:
    print("Creating synthetic practice data because these files are missing:", missing)

    # Stores
    n_stores = 20
    stores_gen = pd.DataFrame({
        "store_id": [f"S{i:03d}" for i in range(1, n_stores + 1)],
        "city": rng.choice(["Coimbatore", "Chennai", "Madurai", "Salem",
                            "Tiruchirappalli", "Erode", "Tirunelveli"], n_stores),
        "format": rng.choice(["Express", "Standard", "Hypermarket"], n_stores, p=[0.30, 0.50, 0.20]),
        "floor_area_sqft": rng.integers(2500, 25001, n_stores)
    })

    # Products
    n_products = 100
    categories = ["Dairy", "Bakery", "Produce", "Beverages", "Snacks",
                  "Household", "Personal Care", "Frozen Foods"]
    products_gen = pd.DataFrame({
        "product_id": [f"P{i:04d}" for i in range(1, n_products + 1)],
        "product_name": [f"Product {i:03d}" for i in range(1, n_products + 1)],
        "category": rng.choice(categories, n_products),
        "brand": rng.choice(["FreshChoice", "DailyFarm", "ValuePlus", "SunHarvest", "HomeEase"], n_products),
        "price_band": rng.choice(["Budget", "Mid-range", "Premium"], n_products, p=[0.40, 0.45, 0.15]),
        "unit_price": np.round(rng.uniform(20, 800, n_products), 2)
    })

    # Promotions
    promotions_gen = pd.DataFrame([
        ("PROMO001", "Weekend Saver", 10, 3),
        ("PROMO002", "Festival Bonanza", 20, 7),
        ("PROMO003", "Buy More Save More", 15, 5),
        ("PROMO004", "Fresh Picks", 5, 4),
        ("PROMO005", "Clearance Week", 25, 7),
    ], columns=["promotion_id", "campaign", "discount_pct", "duration_days"])

    # Calendar
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    festival_map = {
        "2025-01-14": "Pongal",
        "2025-03-14": "Holi",
        "2025-04-14": "Tamil New Year",
        "2025-08-15": "Independence Day",
        "2025-10-20": "Diwali",
        "2025-12-25": "Christmas"
    }
    calendar_gen = pd.DataFrame({"date": dates})
    calendar_gen["weekday"] = calendar_gen["date"].dt.day_name()
    calendar_gen["is_weekend"] = calendar_gen["date"].dt.dayofweek.isin([5, 6])
    calendar_gen["holiday"] = calendar_gen["date"].dt.strftime("%Y-%m-%d").map({
        "2025-01-14": "Pongal", "2025-04-14": "Tamil New Year",
        "2025-08-15": "Independence Day", "2025-10-20": "Diwali",
        "2025-12-25": "Christmas"
    }).fillna("None")
    calendar_gen["festival"] = calendar_gen["date"].dt.strftime("%Y-%m-%d").map(festival_map).fillna("None")
    calendar_gen["season"] = calendar_gen["date"].dt.month.map({
        12: "Winter", 1: "Winter", 2: "Winter",
        3: "Summer", 4: "Summer", 5: "Summer",
        6: "Monsoon", 7: "Monsoon", 8: "Monsoon", 9: "Monsoon",
        10: "Festive", 11: "Festive"
    })

    # Sales transactions
    n_rows = 12000
    sales_dates = pd.to_datetime(rng.choice(dates.to_numpy(), size=n_rows))
    sales_gen = pd.DataFrame({
        "sales_id": [f"T{i:06d}" for i in range(1, n_rows + 1)],
        "date": sales_dates,
        "store_id": rng.choice(stores_gen["store_id"], n_rows),
        "product_id": rng.choice(products_gen["product_id"], n_rows)
    })
    sales_gen = sales_gen.merge(products_gen[["product_id", "unit_price"]], on="product_id", how="left")
    sales_gen = sales_gen.merge(calendar_gen[["date", "is_weekend", "festival"]], on="date", how="left")
    sales_gen["units"] = rng.poisson(4, n_rows) + 1
    weekend_mask = sales_gen["is_weekend"].to_numpy()
    sales_gen.loc[weekend_mask, "units"] += rng.binomial(3, 0.35, weekend_mask.sum())
    festival_mask = sales_gen["festival"].ne("None").to_numpy()
    sales_gen.loc[festival_mask, "units"] += rng.integers(1, 6, festival_mask.sum())
    promo_mask = rng.random(n_rows) < 0.28
    sales_gen["promotion_id"] = pd.Series([None] * n_rows, dtype="object")
    sales_gen.loc[promo_mask, "promotion_id"] = rng.choice(
        promotions_gen["promotion_id"], promo_mask.sum()
    )
    discount_lookup = promotions_gen.set_index("promotion_id")["discount_pct"].to_dict()
    sales_gen["discount_pct"] = sales_gen["promotion_id"].map(discount_lookup).fillna(0)
    sales_gen["revenue"] = np.round(
        sales_gen["units"] * sales_gen["unit_price"] *
        (1 - sales_gen["discount_pct"] / 100), 2
    )
    sales_gen["date"] = sales_gen["date"].dt.strftime("%Y-%m-%d")
    sales_gen = sales_gen[["sales_id", "date", "store_id", "product_id",
                           "units", "revenue", "promotion_id", "discount_pct"]]

    generated = {
        "sales.csv": sales_gen,
        "products.csv": products_gen,
        "stores.csv": stores_gen,
        "promotions.csv": promotions_gen,
        "calendar.csv": calendar_gen
    }
    for filename, frame in generated.items():
        frame.to_csv(DATA_DIR / filename, index=False)

    (DATA_DIR / "README.txt").write_text(
        "Synthetic FreshCart practice data. Not real business data.\n"
        "Join keys: sales.product_id -> products.product_id; "
        "sales.store_id -> stores.store_id; sales.promotion_id -> promotions.promotion_id; "
        "sales.date -> calendar.date.\n",
        encoding="utf-8"
    )

print("\nDataset files available:")
for f in required_files:
    print("-", f, "->", locate_csv(f))

## 2. Load and understand the datasets
This section displays dataset dimensions, column names, data types, and sample rows.

In [ ]:
# 2A. Load all five datasets
paths = {name.replace(".csv", ""): locate_csv(name) for name in required_files}
sales = pd.read_csv(paths["sales"])
products = pd.read_csv(paths["products"])
stores = pd.read_csv(paths["stores"])
promotions = pd.read_csv(paths["promotions"])
calendar = pd.read_csv(paths["calendar"])

datasets = {
    "sales": sales,
    "products": products,
    "stores": stores,
    "promotions": promotions,
    "calendar": calendar
}

overview_rows = []
for name, df in datasets.items():
    overview_rows.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum())
    })
overview = pd.DataFrame(overview_rows)
display(overview)

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Columns:", df.columns.tolist())
    print("Data types:")
    display(df.dtypes.to_frame("dtype"))
    display(df.head())

## 3. Data quality inspection
The report checks missing values, duplicate rows, numeric summaries, invalid sales quantities/revenue, and invalid relationships between files. Missing promotion IDs can be valid when no promotion was used.

In [ ]:
# 3A. Quality report for every dataset
quality_rows = []
for name, df in datasets.items():
    for col in df.columns:
        quality_rows.append({
            "dataset": name,
            "column": col,
            "dtype": str(df[col].dtype),
            "missing_count": int(df[col].isna().sum()),
            "missing_percent": round(float(df[col].isna().mean() * 100), 2),
            "distinct_values": int(df[col].nunique(dropna=True))
        })
quality_report = pd.DataFrame(quality_rows)
display(quality_report)

print("Exact duplicate rows by dataset:")
display(pd.DataFrame({
    "dataset": list(datasets.keys()),
    "duplicate_rows": [int(df.duplicated().sum()) for df in datasets.values()]
}))

print("Numeric summaries:")
for name, df in datasets.items():
    numeric = df.select_dtypes(include="number")
    if not numeric.empty:
        print(f"\n{name}:")
        display(numeric.describe().T)

In [ ]:
# 3B. Sales and relationship validation
sales_check = sales.copy()
sales_check["date"] = pd.to_datetime(sales_check["date"], errors="coerce")

validation = {
    "duplicate sales_id": int(sales_check["sales_id"].duplicated().sum()),
    "invalid/missing sales dates": int(sales_check["date"].isna().sum()),
    "missing units": int(sales_check["units"].isna().sum()),
    "zero or negative units": int((pd.to_numeric(sales_check["units"], errors="coerce") <= 0).sum()),
    "missing revenue": int(sales_check["revenue"].isna().sum()),
    "negative revenue": int((pd.to_numeric(sales_check["revenue"], errors="coerce") < 0).sum()),
    "unknown product IDs": int((~sales_check["product_id"].isin(products["product_id"])).sum()),
    "unknown store IDs": int((~sales_check["store_id"].isin(stores["store_id"])).sum()),
    "unknown nonmissing promotion IDs": int((sales_check["promotion_id"].notna() & ~sales_check["promotion_id"].isin(promotions["promotion_id"])).sum()),
    "duplicate calendar dates": int(pd.to_datetime(calendar["date"], errors="coerce").duplicated().sum()),
    "sales dates missing from calendar": int((~sales_check["date"].isin(pd.to_datetime(calendar["date"], errors="coerce"))).sum())
}
display(pd.DataFrame(validation.items(), columns=["check", "count"]))

print("Product categories:")
display(products["category"].value_counts(dropna=False).to_frame("product_count"))

## 4. Clean and combine the datasets
Cleaning is conservative: exact duplicate rows are removed, date and numeric types are standardized, category/label whitespace is trimmed, and unusable sales records are excluded. Records with no promotion ID are retained because they can represent non-promotional sales.

In [ ]:
# 4A. Clean copies
sales_clean = sales.copy()
products_clean = products.copy()
stores_clean = stores.copy()
promotions_clean = promotions.copy()
calendar_clean = calendar.copy()

# Remove exact duplicate rows
sales_clean = sales_clean.drop_duplicates().copy()
products_clean = products_clean.drop_duplicates().copy()
stores_clean = stores_clean.drop_duplicates().copy()
promotions_clean = promotions_clean.drop_duplicates().copy()
calendar_clean = calendar_clean.drop_duplicates().copy()

# Standardize date and numeric columns
sales_clean["date"] = pd.to_datetime(sales_clean["date"], errors="coerce")
calendar_clean["date"] = pd.to_datetime(calendar_clean["date"], errors="coerce")
for col in ["units", "revenue", "discount_pct"]:
    if col in sales_clean.columns:
        sales_clean[col] = pd.to_numeric(sales_clean[col], errors="coerce")

# Normalize text labels
for col in ["category", "brand", "price_band"]:
    if col in products_clean.columns:
        products_clean[col] = products_clean[col].astype("string").str.strip()
products_clean["category"] = products_clean["category"].str.title()
for df in [stores_clean, promotions_clean, calendar_clean]:
    for col in df.select_dtypes(include=["object", "string"]).columns:
        df[col] = df[col].astype("string").str.strip()

# Exclude records with essential invalid values
invalid_sales_mask = (
    sales_clean["date"].isna()
    | sales_clean["units"].isna()
    | sales_clean["revenue"].isna()
    | (sales_clean["units"] <= 0)
    | (sales_clean["revenue"] < 0)
)
invalid_sales_count = int(invalid_sales_mask.sum())
sales_clean = sales_clean.loc[~invalid_sales_mask].copy()

# Deduplicate dimension tables by their join keys, keeping first record.
# The report above should be reviewed if real data contains conflicting duplicates.
for frame, key in [
    (products_clean, "product_id"),
    (stores_clean, "store_id"),
    (promotions_clean, "promotion_id"),
    (calendar_clean, "date")
]:
    if key in frame.columns:
        frame.drop_duplicates(subset=[key], keep="first", inplace=True)

print("Invalid sales rows excluded:", invalid_sales_count)
print("Cleaned row counts:")
for name, df in {
    "sales": sales_clean, "products": products_clean, "stores": stores_clean,
    "promotions": promotions_clean, "calendar": calendar_clean
}.items():
    print(f"{name}: {len(df):,} rows")

In [ ]:
# 4B. Merge into a master analysis dataset
master_df = sales_clean.merge(
    products_clean, on="product_id", how="left", validate="many_to_one"
)
master_df = master_df.merge(
    stores_clean, on="store_id", how="left", validate="many_to_one"
)
master_df = master_df.merge(
    promotions_clean, on="promotion_id", how="left", validate="many_to_one",
    suffixes=("", "_promotion")
)
master_df = master_df.merge(
    calendar_clean, on="date", how="left", validate="many_to_one",
    suffixes=("", "_calendar")
)

print("Sales rows before merge:", len(sales_clean))
print("Master rows after merge:", len(master_df))
print("Master dataset shape:", master_df.shape)
print("Columns:", master_df.columns.tolist())
display(master_df.head())

# Save master dataset
master_path = OUTPUT_DIR / "freshcart_master_dataset.csv"
master_df.to_csv(master_path, index=False)
print("Saved:", master_path)

## 5. Exploratory Data Analysis (EDA)
The charts and tables below address the use-case questions. Since the data is synthetic, treat findings as practice insights rather than real business conclusions.

In [ ]:
# 5A. Prepare EDA fields
eda_df = master_df.copy()
eda_df["date"] = pd.to_datetime(eda_df["date"], errors="coerce")
eda_df["revenue"] = pd.to_numeric(eda_df["revenue"], errors="coerce")
eda_df["units"] = pd.to_numeric(eda_df["units"], errors="coerce")
eda_df = eda_df.dropna(subset=["date", "revenue", "units"]).copy()
eda_df["month"] = eda_df["date"].dt.month
eda_df["month_name"] = eda_df["date"].dt.month_name()
eda_df["day_name"] = eda_df["date"].dt.day_name()

print("EDA dataset shape:", eda_df.shape)
print("Date range:", eda_df["date"].min().date(), "to", eda_df["date"].max().date())
display(eda_df.head())

In [ ]:
# 5B. Which categories drive revenue?
category_revenue = (eda_df.groupby("category", dropna=False)["revenue"]
                    .sum().sort_values(ascending=False))
display(category_revenue.rename("total_revenue").to_frame())

ax = category_revenue.sort_values().plot(kind="barh", figsize=(9, 5))
ax.set_title("Total Revenue by Product Category")
ax.set_xlabel("Revenue")
ax.set_ylabel("Category")
plt.tight_layout()
plt.savefig(CHART_DIR / "01_category_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# 5C. Store performance and monthly trend
monthly_revenue = (eda_df.groupby(pd.Grouper(key="date", freq="MS"))["revenue"]
                   .sum().rename("revenue").reset_index())
display(monthly_revenue)

ax = monthly_revenue.plot(x="date", y="revenue", marker="o", figsize=(10, 4), legend=False)
ax.set_title("Monthly Revenue Across All Stores")
ax.set_xlabel("Month")
ax.set_ylabel("Revenue")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(CHART_DIR / "02_monthly_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

store_revenue = (eda_df.groupby(["store_id", "city"], dropna=False)["revenue"]
                 .sum().sort_values(ascending=False).head(10))
display(store_revenue.rename("total_revenue").to_frame())

ax = store_revenue.sort_values().plot(kind="barh", figsize=(9, 5))
ax.set_title("Top 10 Stores by Revenue")
ax.set_xlabel("Revenue")
ax.set_ylabel("Store / City")
plt.tight_layout()
plt.savefig(CHART_DIR / "03_top_stores.png", dpi=150, bbox_inches="tight")
plt.show()

# First-vs-last observed month comparison, not year-over-year growth
store_month = eda_df.groupby(["store_id", pd.Grouper(key="date", freq="MS")])["revenue"].sum().reset_index()
growth_rows = []
for store_id, group in store_month.sort_values("date").groupby("store_id"):
    group = group.sort_values("date")
    nonzero = group[group["revenue"] > 0]
    if len(nonzero) >= 2:
        first = float(nonzero.iloc[0]["revenue"])
        last = float(nonzero.iloc[-1]["revenue"])
        growth_rows.append({
            "store_id": store_id,
            "first_observed_month_revenue": first,
            "last_observed_month_revenue": last,
            "change_percent": ((last - first) / first * 100) if first else np.nan
        })
store_growth = pd.DataFrame(growth_rows).sort_values("change_percent", ascending=False)
print("Store change from first to last observed month (not year-over-year):")
display(store_growth.head(10))

In [ ]:
# 5D. Promotion comparison
# Promotion IDs can be missing for non-promotional sales.
eda_df["promotion_status"] = np.where(
    eda_df["promotion_id"].notna(), "With Promotion", "Without Promotion"
)
promotion_analysis = eda_df.groupby("promotion_status").agg(
    sales_records=("sales_id", "count"),
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
    average_revenue_per_record=("revenue", "mean"),
    average_units_per_record=("units", "mean")
).round(2)
display(promotion_analysis)

ax = promotion_analysis["average_revenue_per_record"].plot(kind="bar", figsize=(7, 4))
ax.set_title("Average Revenue per Record: Promotion vs No Promotion")
ax.set_xlabel("")
ax.set_ylabel("Average revenue per sales record")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.savefig(CHART_DIR / "04_promotion_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

print("Interpretation caution: this is descriptive; differences do not prove promotions caused the change.")

In [ ]:
# 5E. Festival impact: compare average daily sales on festival vs non-festival dates
festival_daily = eda_df.groupby(["date", "festival"], dropna=False).agg(
    daily_units=("units", "sum"),
    daily_revenue=("revenue", "sum")
).reset_index()
festival_daily["festival_flag"] = np.where(
    festival_daily["festival"].fillna("None").astype(str).str.lower().eq("none"),
    "Non-festival day", "Festival day"
)
festival_comparison = festival_daily.groupby("festival_flag").agg(
    days=("date", "nunique"),
    average_daily_units=("daily_units", "mean"),
    average_daily_revenue=("daily_revenue", "mean"),
    total_units=("daily_units", "sum"),
    total_revenue=("daily_revenue", "sum")
).round(2)
display(festival_comparison)

named_festivals = festival_daily[festival_daily["festival_flag"] == "Festival day"]
if not named_festivals.empty:
    by_festival = named_festivals.groupby("festival").agg(
        observed_days=("date", "nunique"),
        average_daily_units=("daily_units", "mean"),
        average_daily_revenue=("daily_revenue", "mean")
    ).sort_values("average_daily_units", ascending=False)
    display(by_festival)

    ax = by_festival["average_daily_units"].sort_values().plot(kind="barh", figsize=(9, 4))
    ax.set_title("Average Units Sold on Named Festival Dates")
    ax.set_xlabel("Average units per observed date")
    ax.set_ylabel("Festival")
    plt.tight_layout()
    plt.savefig(CHART_DIR / "05_festival_demand.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# 5F. Seasonal demand and peak shopping periods
season_analysis = eda_df.groupby("season", dropna=False).agg(
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
    sales_records=("sales_id", "count"),
    average_revenue_per_record=("revenue", "mean")
).sort_values("total_revenue", ascending=False)
display(season_analysis)

weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_analysis = eda_df.groupby("day_name").agg(
    total_revenue=("revenue", "sum"),
    average_revenue_per_record=("revenue", "mean"),
    total_units=("units", "sum"),
    sales_records=("sales_id", "count")
).reindex(weekday_order)
display(weekday_analysis)

ax = weekday_analysis["total_revenue"].plot(kind="bar", figsize=(9, 4))
ax.set_title("Total Revenue by Weekday")
ax.set_xlabel("Weekday")
ax.set_ylabel("Revenue")
ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.savefig(CHART_DIR / "06_weekday_revenue.png", dpi=150, bbox_inches="tight")
plt.show()

# Product seasonality: top products by variation in monthly units (minimum data coverage)
product_month = eda_df.groupby(["product_id", "month"])["units"].sum().reset_index()
seasonality = product_month.groupby("product_id")["units"].agg(["mean", "std", "min", "max"])
seasonality["coefficient_of_variation"] = seasonality["std"] / seasonality["mean"].replace(0, np.nan)
seasonality = seasonality.sort_values("coefficient_of_variation", ascending=False)
print("Products with highest monthly demand variation (interpret cautiously; sparse records can inflate variation):")
display(seasonality.head(10))

## 6. Feature engineering
Create date, weekend, promotion, revenue-per-unit, and discount indicator features. These features are for analysis and future modeling practice.

In [ ]:
# 6A. Create engineered features
feature_df = eda_df.copy()
feature_df["year"] = feature_df["date"].dt.year
feature_df["month"] = feature_df["date"].dt.month
feature_df["day"] = feature_df["date"].dt.day
feature_df["day_of_week"] = feature_df["date"].dt.dayofweek
feature_df["week_of_year"] = feature_df["date"].dt.isocalendar().week.astype("Int64")
feature_df["is_weekend"] = (feature_df["day_of_week"] >= 5).astype(int)
feature_df["has_promotion"] = feature_df["promotion_id"].notna().astype(int)

# Ensure discount_pct exists. If absent after a custom upload, recover from promotion table.
if "discount_pct" not in feature_df.columns:
    if "discount_pct" in sales_clean.columns:
        feature_df = feature_df.merge(
            sales_clean[["sales_id", "discount_pct"]], on="sales_id",
            how="left", validate="one_to_one"
        )
    else:
        feature_df["discount_pct"] = 0
feature_df["discount_pct"] = pd.to_numeric(feature_df["discount_pct"], errors="coerce").fillna(0)
feature_df["is_discounted"] = (feature_df["discount_pct"] > 0).astype(int)
feature_df["revenue_per_unit"] = np.where(
    feature_df["units"] > 0, feature_df["revenue"] / feature_df["units"], np.nan
)

print("Feature engineering complete.")
print("Shape:", feature_df.shape)
display(feature_df[[
    c for c in ["date", "units", "revenue", "month", "day_of_week",
                "is_weekend", "has_promotion", "discount_pct",
                "is_discounted", "revenue_per_unit"] if c in feature_df.columns
]].head())

In [ ]:
# 6B. One-hot encode selected categorical variables
encoded_df = feature_df.copy()
categorical_candidates = ["category", "brand", "format", "season", "day_name", "price_band"]
categorical_columns = [c for c in categorical_candidates if c in encoded_df.columns]

encoded_df = pd.get_dummies(
    encoded_df,
    columns=categorical_columns,
    dummy_na=False,
    dtype=int
)

print("Categorical columns encoded:", categorical_columns)
print("Feature dataset shape:", feature_df.shape)
print("Encoded dataset shape:", encoded_df.shape)
display(encoded_df.head())

In [ ]:
# 6C. Preliminary feature selection using correlations
# Revenue is the analysis target. Exclude identifiers and text columns.
candidate_features = [
    "units", "discount_pct", "month", "day_of_week", "is_weekend",
    "has_promotion", "is_discounted", "unit_price", "floor_area_sqft"
]
candidate_features = [c for c in candidate_features if c in feature_df.columns]
selected_df = feature_df[candidate_features + ["revenue"]].copy()

correlation = selected_df.corr(numeric_only=True)["revenue"].drop("revenue").sort_values(ascending=False)
print("Candidate feature correlations with revenue (descriptive, not causal):")
display(correlation.to_frame("correlation_with_revenue"))

ax = correlation.sort_values().plot(kind="barh", figsize=(8, 4))
ax.set_title("Candidate Feature Correlation with Revenue")
ax.set_xlabel("Pearson correlation")
ax.set_ylabel("Feature")
plt.tight_layout()
plt.savefig(CHART_DIR / "07_feature_correlations.png", dpi=150, bbox_inches="tight")
plt.show()

print("Note: revenue is constructed from units, unit price, and discount in the synthetic data, so those relationships are partly built into the dataset.")

## 7. Create findings and save Round 2 deliverables
This section exports the master dataset, engineered dataset, encoded dataset, selected feature table, quality report, EDA summary tables, charts, and a human-readable report.

In [ ]:
# 7A. Save all tables
quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)
master_df.to_csv(OUTPUT_DIR / "freshcart_master_dataset.csv", index=False)
feature_df.to_csv(OUTPUT_DIR / "freshcart_engineered_dataset.csv", index=False)
encoded_df.to_csv(OUTPUT_DIR / "freshcart_encoded_dataset.csv", index=False)
selected_df.to_csv(OUTPUT_DIR / "freshcart_selected_features.csv", index=False)
category_revenue.rename("total_revenue").to_csv(OUTPUT_DIR / "category_revenue.csv")
monthly_revenue.to_csv(OUTPUT_DIR / "monthly_revenue.csv", index=False)
promotion_analysis.to_csv(OUTPUT_DIR / "promotion_analysis.csv")
season_analysis.to_csv(OUTPUT_DIR / "season_analysis.csv")
weekday_analysis.to_csv(OUTPUT_DIR / "weekday_analysis.csv")
if "store_growth" in globals():
    store_growth.to_csv(OUTPUT_DIR / "store_month_change.csv", index=False)

# Findings from observed data, generated dynamically
top_category = category_revenue.index[0] if len(category_revenue) else "Not available"
top_category_revenue = float(category_revenue.iloc[0]) if len(category_revenue) else float("nan")
top_store = eda_df.groupby("store_id")["revenue"].sum().idxmax() if not eda_df.empty else "Not available"
top_store_revenue = float(eda_df.groupby("store_id")["revenue"].sum().max()) if not eda_df.empty else float("nan")
peak_month_row = monthly_revenue.loc[monthly_revenue["revenue"].idxmax()] if not monthly_revenue.empty else None
peak_month = peak_month_row["date"].strftime("%B %Y") if peak_month_row is not None else "Not available"
peak_weekday = weekday_analysis["total_revenue"].idxmax() if weekday_analysis["total_revenue"].notna().any() else "Not available"
peak_season = season_analysis["total_revenue"].idxmax() if not season_analysis.empty else "Not available"

report = f'''FRESHCART SUPERMARKET SALES INSIGHT — ROUND 2 REPORT
====================================================

Dataset note
------------
This report uses synthetic practice data. It is not real FreshCart business data.

Dataset dimensions
------------------
Sales input rows: {len(sales):,}
Clean sales rows: {len(sales_clean):,}
Master dataset rows: {len(master_df):,}
Master dataset columns: {master_df.shape[1]}

Data preparation
----------------
Exact duplicate rows were removed from each input.
Dates and numeric sales columns were standardized.
Sales rows with missing/invalid dates, missing units/revenue, non-positive units, or negative revenue were excluded.
Text labels were trimmed and product categories standardized.
The five datasets were combined with left joins and many-to-one validation.

Key descriptive findings
------------------------
Top revenue category: {top_category} (revenue {top_category_revenue:,.2f})
Highest revenue store: {top_store} (revenue {top_store_revenue:,.2f})
Peak month by total revenue: {peak_month}
Highest revenue weekday: {peak_weekday}
Highest revenue season: {peak_season}

Interpretation cautions
-----------------------
- These results describe the generated synthetic data only.
- Promotion differences are observational and do not establish causation.
- Store month comparison is first-to-last observed month, not year-over-year growth.
- Product seasonality estimates can be unstable when monthly observations are sparse.
- Revenue correlations are not proof of independent influence; revenue is partly constructed from units, price, and discounts.

Files included in outputs
-------------------------
freshcart_master_dataset.csv
freshcart_engineered_dataset.csv
freshcart_encoded_dataset.csv
freshcart_selected_features.csv
data_quality_report.csv
category_revenue.csv
monthly_revenue.csv
promotion_analysis.csv
season_analysis.csv
weekday_analysis.csv
store_month_change.csv (when available)
charts/*.png
'''
(OUTPUT_DIR / "round2_summary_report.txt").write_text(report, encoding="utf-8")
print(report)

In [ ]:
# 7B. Package all outputs into one downloadable ZIP
zip_output = BASE_DIR / "FreshCart_Round2_Deliverables.zip"
with zipfile.ZipFile(zip_output, "w", zipfile.ZIP_DEFLATED) as zf:
    for file_path in OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zf.write(file_path, arcname=str(file_path.relative_to(OUTPUT_DIR)))

print("Deliverables packaged:", zip_output)
print("Files included:")
for file_path in sorted(OUTPUT_DIR.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(OUTPUT_DIR))

# Download the deliverables ZIP in Google Colab
try:
    from google.colab import files
    files.download(str(zip_output))
except Exception:
    print("ZIP is ready at:", zip_output)

## 8. Final Round 2 checklist
- [x] Load and understand datasets
- [x] Inspect missing values, duplicates, data types, unusual values, and key relationships
- [x] Clean and combine the datasets
- [x] Create EDA tables and charts for category, store, promotions, festivals, seasons, and weekdays
- [x] Engineer time-based and promotion-related features
- [x] Encode selected categorical variables
- [x] Perform preliminary feature selection using correlation
- [x] Export datasets, charts, quality report, and summary report

### Before submission
1. Run every cell from top to bottom without skipping.
2. Review the actual outputs and replace generic presentation statements with findings from your generated data.
3. Keep the synthetic-data disclaimer in your submission unless your instructor gives you real datasets.
4. Your assessment also includes **Round 3 (Excel analysis)** and **Round 4 (business pitch)**, which are separate deliverables and are not completed by this Round 2 notebook.

# Round 3 — Excel Analysis and Business Dashboard Tables

This section creates an Excel workbook with separate sheets for the cleaned master data, KPI summary, category/store/month analysis, promotions, festivals, seasons, and weekday patterns. The workbook is generated from the data loaded and cleaned above.

In [ ]:
# 8A. Create an Excel workbook for Round 3
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

excel_path = OUTPUT_DIR / "FreshCart_Excel_Analysis.xlsx"

# Build KPI and analysis tables from the current dataset
total_revenue = float(eda_df["revenue"].sum())
total_units = int(eda_df["units"].sum())
avg_transaction_revenue = float(eda_df["revenue"].mean())
unique_stores = int(eda_df["store_id"].nunique())
unique_products = int(eda_df["product_id"].nunique())
promotion_share = float(eda_df["promotion_id"].notna().mean() * 100)

kpi_summary = pd.DataFrame({
    "KPI": ["Sales records", "Total revenue", "Total units sold", "Average revenue per sales record",
            "Number of stores", "Number of products", "Sales records with promotion (%)"],
    "Value": [len(eda_df), total_revenue, total_units, avg_transaction_revenue,
              unique_stores, unique_products, promotion_share]
})

store_summary = (eda_df.groupby(["store_id", "city", "format"], dropna=False)
                 .agg(total_revenue=("revenue", "sum"),
                      total_units=("units", "sum"),
                      sales_records=("sales_id", "count"))
                 .reset_index().sort_values("total_revenue", ascending=False))
category_summary = (eda_df.groupby("category", dropna=False)
                    .agg(total_revenue=("revenue", "sum"),
                         total_units=("units", "sum"),
                         sales_records=("sales_id", "count"))
                    .reset_index().sort_values("total_revenue", ascending=False))
monthly_summary = monthly_revenue.copy()
monthly_summary["date"] = pd.to_datetime(monthly_summary["date"]).dt.strftime("%Y-%m")
promotion_summary = promotion_analysis.reset_index()
season_summary = season_analysis.reset_index()
weekday_summary = weekday_analysis.reset_index()
festival_summary = festival_daily.reset_index()

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    kpi_summary.to_excel(writer, sheet_name="KPI Summary", index=False)
    category_summary.to_excel(writer, sheet_name="Category Analysis", index=False)
    store_summary.to_excel(writer, sheet_name="Store Analysis", index=False)
    monthly_summary.to_excel(writer, sheet_name="Monthly Trend", index=False)
    promotion_summary.to_excel(writer, sheet_name="Promotion Analysis", index=False)
    festival_summary.to_excel(writer, sheet_name="Festival Analysis", index=False)
    season_summary.to_excel(writer, sheet_name="Season Analysis", index=False)
    weekday_summary.to_excel(writer, sheet_name="Weekday Analysis", index=False)
    quality_report.to_excel(writer, sheet_name="Data Quality", index=False)
    master_df.to_excel(writer, sheet_name="Master Data", index=False)

# Apply readable formatting to each sheet
from openpyxl import load_workbook
wb = load_workbook(excel_path)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    ws.auto_filter.ref = ws.dimensions
    for cell in ws[1]:
        cell.font = Font(bold=True, color="FFFFFF")
        cell.fill = PatternFill("solid", fgColor="245A81")
        cell.alignment = Alignment(wrap_text=True, vertical="center")
    ws.row_dimensions[1].height = 28
    for col_cells in ws.columns:
        max_len = 0
        col_letter = get_column_letter(col_cells[0].column)
        for cell in col_cells[:300]:  # limit width scan for large master sheet
            try:
                max_len = max(max_len, len(str(cell.value)) if cell.value is not None else 0)
            except Exception:
                pass
        ws.column_dimensions[col_letter].width = min(max(max_len + 2, 12), 32)
wb.save(excel_path)
print("Excel workbook saved:", excel_path)
print("Sheets:", ["KPI Summary", "Category Analysis", "Store Analysis", "Monthly Trend",
                  "Promotion Analysis", "Festival Analysis", "Season Analysis", "Weekday Analysis",
                  "Data Quality", "Master Data"])
display(kpi_summary)


# Round 2 — Predictive Modeling

We predict **revenue**, a continuous numeric value, with a Random Forest Regressor. Therefore, accuracy and F1-score are **not applicable** to this regression model. We evaluate it using R², MAE, and RMSE. Because this notebook may use synthetic practice data, performance values must not be presented as real FreshCart business results.

In [ ]:
# 9A. Train and evaluate a revenue prediction model
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

model_df = feature_df.copy()
model_df["revenue"] = pd.to_numeric(model_df["revenue"], errors="coerce")
model_df = model_df.dropna(subset=["revenue"]).copy()

# Use predictors available at prediction time; do not use revenue-derived columns.
numeric_candidates = ["units", "unit_price", "discount_pct", "month", "day_of_week",
                      "is_weekend", "has_promotion", "is_discounted", "floor_area_sqft"]
categorical_candidates = ["category", "brand", "format", "season", "price_band", "city"]
numeric_features = [c for c in numeric_candidates if c in model_df.columns]
categorical_features = [c for c in categorical_candidates if c in model_df.columns]
features = numeric_features + categorical_features
X = model_df[features].copy()
y = model_df["revenue"].astype(float)

# Convert nullable booleans / strings safely for sklearn
for c in numeric_features:
    X[c] = pd.to_numeric(X[c], errors="coerce")
for c in categorical_features:
    X[c] = X[c].astype("object").where(X[c].notna(), np.nan)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=150, random_state=SEED, n_jobs=-1, min_samples_leaf=2
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
mae = float(mean_absolute_error(y_test, y_pred))
r2 = float(r2_score(y_test, y_pred))
model_metrics = pd.DataFrame([
    {"Metric": "Training records", "Value": len(X_train)},
    {"Metric": "Test records", "Value": len(X_test)},
    {"Metric": "R² score", "Value": r2},
    {"Metric": "RMSE", "Value": rmse},
    {"Metric": "MAE", "Value": mae},
    {"Metric": "Accuracy", "Value": "Not applicable — regression"},
    {"Metric": "F1-score", "Value": "Not applicable — regression"}
])
display(model_metrics)
print("Model features:", features)
print("Important note: synthetic data may make performance look unrealistically strong.")


In [ ]:
# 9B. Visualize actual versus predicted revenue and residuals
plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.45)
lims = [min(float(y_test.min()), float(np.min(y_pred))), max(float(y_test.max()), float(np.max(y_pred)))]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Actual revenue")
plt.ylabel("Predicted revenue")
plt.title("Actual vs Predicted Revenue")
plt.tight_layout()
plt.savefig(CHART_DIR / "08_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

residuals = y_test.to_numpy() - y_pred
plt.figure(figsize=(7, 4))
plt.hist(residuals, bins=30)
plt.xlabel("Prediction error (actual - predicted)")
plt.ylabel("Frequency")
plt.title("Distribution of Revenue Prediction Errors")
plt.tight_layout()
plt.savefig(CHART_DIR / "09_prediction_errors.png", dpi=150, bbox_inches="tight")
plt.show()

prediction_results = pd.DataFrame({
    "actual_revenue": y_test.to_numpy(),
    "predicted_revenue": y_pred,
    "absolute_error": np.abs(y_test.to_numpy() - y_pred)
})
prediction_results.to_csv(OUTPUT_DIR / "revenue_model_predictions.csv", index=False)
model_metrics.to_csv(OUTPUT_DIR / "model_evaluation_metrics.csv", index=False)
print("Saved prediction-level results and model metrics.")


# Round 3 — Business Findings and Decision Support

The following statements are calculated from the current dataset. Use the displayed outputs rather than inventing figures. These are descriptive findings, not proof that promotions or festivals caused a change in sales.

In [ ]:
# 10A. Automatically generate the main findings
top_cat_name = str(category_summary.iloc[0]["category"]) if len(category_summary) else "Unavailable"
top_cat_rev = float(category_summary.iloc[0]["total_revenue"]) if len(category_summary) else float("nan")
top_store_name = str(store_summary.iloc[0]["store_id"]) if len(store_summary) else "Unavailable"
top_store_rev = float(store_summary.iloc[0]["total_revenue"]) if len(store_summary) else float("nan")
peak_month_name = str(monthly_summary.loc[monthly_summary["revenue"].idxmax(), "date"]) if len(monthly_summary) else "Unavailable"
peak_weekday_name = str(weekday_summary.loc[weekday_summary["total_revenue"].idxmax(), "day_name"]) if len(weekday_summary) else "Unavailable"
peak_season_name = str(season_summary.loc[season_summary["total_revenue"].idxmax(), "season"]) if len(season_summary) else "Unavailable"

with_promo = eda_df.loc[eda_df["promotion_id"].notna(), "revenue"]
without_promo = eda_df.loc[eda_df["promotion_id"].isna(), "revenue"]
promo_comparison = pd.DataFrame({
    "group": ["With promotion", "Without promotion"],
    "average_revenue_per_record": [
        float(with_promo.mean()) if len(with_promo) else np.nan,
        float(without_promo.mean()) if len(without_promo) else np.nan
    ],
    "records": [int(len(with_promo)), int(len(without_promo))]
})
display(promo_comparison)

findings = [
    f"Highest-revenue product category: {top_cat_name} (total revenue {top_cat_rev:,.2f}).",
    f"Highest-revenue store: {top_store_name} (total revenue {top_store_rev:,.2f}).",
    f"Peak month in the dataset: {peak_month_name}.",
    f"Highest-revenue weekday: {peak_weekday_name}.",
    f"Highest-revenue season: {peak_season_name}.",
    "Promotion comparisons are observational; they do not establish that discounts caused higher revenue.",
    "All values are based on the current loaded dataset. If synthetic data is used, results are for demonstration only."
]
for item in findings:
    print("•", item)

findings_path = OUTPUT_DIR / "business_findings.txt"
findings_path.write_text("\n".join(findings), encoding="utf-8")


# Round 4 — Pitch Presentation Content

Use this as a short presentation script. The notebook calculates the leading category/store/month dynamically. Replace synthetic-data wording only if you later load an authorized real dataset.

In [ ]:
# 11A. Generate a ready-to-use pitch script and assessment notes
pitch = f'''FRESHCART SUPERMARKET SALES INSIGHT — PROJECT PITCH

Problem
FreshCart needs a clear view of how product categories, stores, promotions, calendar events, and seasons relate to sales. Without a combined view, inventory and promotional decisions may rely on assumptions.

Approach
We inspected five related datasets: sales, products, stores, promotions, and calendar. We checked data types, missing values, duplicates, invalid quantities/revenue, and key relationships. We cleaned the data and joined it into one analysis table.

Analysis
We compared revenue by product category and store, tracked monthly sales, and examined promotion, festival, seasonal, and weekday patterns. We engineered calendar and promotion features and encoded categorical variables.

Key findings from this run
- Highest-revenue category: {top_cat_name} (revenue {top_cat_rev:,.2f})
- Highest-revenue store: {top_store_name} (revenue {top_store_rev:,.2f})
- Peak month: {peak_month_name}
- Highest-revenue weekday: {peak_weekday_name}
- Highest-revenue season: {peak_season_name}

Predictive model
A Random Forest Regressor was trained with an 80:20 train-test split to estimate revenue. R² = {r2:.4f}, RMSE = {rmse:.2f}, and MAE = {mae:.2f} for this run. Accuracy and F1-score are not applicable because revenue is a continuous target.

Recommendations
1. Review stock and shelf allocation for high-revenue categories.
2. Compare store-level performance and investigate low-performing stores before changing operations.
3. Plan inventory ahead of observed peak periods and festival dates.
4. Evaluate promotions using controlled comparisons where possible, including margin and units sold—not revenue alone.
5. Re-train and validate the model on genuine, representative business data before using it operationally.

Limitations
The included dataset is synthetic practice data unless replaced with official data. Relationships in synthetic data may be intentionally constructed, and observed associations do not prove causation.
'''

pitch_path = OUTPUT_DIR / "FreshCart_Pitch_and_Report.txt"
pitch_path.write_text(pitch, encoding="utf-8")
print(pitch)

assessment_notes = pd.DataFrame([
    ["Problem statement", "Analyze the influence of product, store, promotion, festival, and seasonal factors on supermarket sales."],
    ["Data sources", "sales.csv, products.csv, stores.csv, promotions.csv, calendar.csv."],
    ["Preprocessing", "Inspect missing values, duplicates, types, invalid values, standardize labels, remove invalid records, merge datasets."],
    ["EDA", "Category, store, monthly, promotion, festival, season, and weekday analysis."],
    ["Feature engineering", "Year, month, day, weekday, weekend flag, promotion flag, discount flag, revenue per unit."],
    ["Model", "Random Forest Regressor to predict continuous revenue."],
    ["Evaluation", f"R²={r2:.4f}; RMSE={rmse:.2f}; MAE={mae:.2f}. Accuracy/F1 are not applicable for regression."],
    ["Business use", "Support category/store review and planning around observed demand patterns."],
    ["Limitation", "Synthetic data; findings and metrics are not real-world FreshCart results."]
], columns=["Topic", "Answer"])
assessment_notes.to_csv(OUTPUT_DIR / "assessment_notes.csv", index=False)
display(assessment_notes)


# Final — Package every project deliverable

Run this last cell after all previous cells succeed. It creates one ZIP containing the cleaned datasets, EDA outputs, charts, Excel workbook, model metrics, prediction results, findings, and pitch notes. In Google Colab, the ZIP download will start automatically.

In [ ]:
# 12. Final project bundle
# Save all key summary tables again so the package is self-contained.
quality_report.to_csv(OUTPUT_DIR / "data_quality_report.csv", index=False)
master_df.to_csv(OUTPUT_DIR / "freshcart_master_dataset.csv", index=False)
feature_df.to_csv(OUTPUT_DIR / "freshcart_engineered_dataset.csv", index=False)
encoded_df.to_csv(OUTPUT_DIR / "freshcart_encoded_dataset.csv", index=False)
selected_df.to_csv(OUTPUT_DIR / "freshcart_selected_features.csv", index=False)
category_summary.to_csv(OUTPUT_DIR / "category_analysis.csv", index=False)
store_summary.to_csv(OUTPUT_DIR / "store_analysis.csv", index=False)
monthly_summary.to_csv(OUTPUT_DIR / "monthly_analysis.csv", index=False)

final_zip = BASE_DIR / "FreshCart_Full_Project_Deliverables.zip"
with zipfile.ZipFile(final_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fp in sorted(OUTPUT_DIR.rglob("*")):
        if fp.is_file() and fp != final_zip:
            zf.write(fp, arcname=str(fp.relative_to(OUTPUT_DIR)))

print("FULL PROJECT COMPLETE")
print("Excel analysis:", excel_path)
print("Pitch report:", pitch_path)
print("Deliverables ZIP:", final_zip)
print("Output files:", sum(1 for fp in OUTPUT_DIR.rglob("*") if fp.is_file()))

try:
    from google.colab import files
    files.download(str(final_zip))
except Exception:
    print("If not running in Colab, find the ZIP at:", final_zip)
